# WHOOSH Parameter Tuning Examples

This notebook is for business and technical walkthroughs.

It uses tiny OCR examples so you can show exactly how each WHOOSH parameter changes detections:

- `slop`
- `edit_distance`
- `min_fuzzy_term_length`
- `keep_stopwords`
- `stem_words`
- `prefixlength`
- page-level vs keyword-level metrics

Production tuning keeps `stem_words=true`; this notebook only turns stemming off in one section to explain why stemming matters.

## 1. Setup

Run this first. It imports the same code used by parameter tuning.

In [ ]:
from __future__ import annotations

import json
import sys
import tempfile
from itertools import product
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd / "Prod" / "parameter_tuning",
    cwd.parent if cwd.name == "parameter_tuning" else cwd,
]
PARAMETER_TUNING_DIR = next(
    (path for path in candidates if (path / "whoosh_encounter_json_keyword_search.py").exists()),
    None,
)
if PARAMETER_TUNING_DIR is None:
    raise FileNotFoundError("Could not find Prod/parameter_tuning from the current notebook directory.")

sys.path.insert(0, str(PARAMETER_TUNING_DIR))

from whoosh_encounter_json_keyword_search import DetectionSettings, run_keyword_detection
from generate_final_comparison_report import compare_output_dir_vs_gt

print(f"Using parameter tuning code from: {PARAMETER_TUNING_DIR}")

## 2. Helper Functions

These helpers create a temporary OCR JSON file, run WHOOSH, and display the page-level detections.

In [ ]:
def make_encounter_text(pages: dict[int, str]) -> str:
    return "\n".join(
        f"<ocr_service_page_start>{page_number}<ocr_service_page_start>\n{text}"
        for page_number, text in pages.items()
    )


def run_demo(
    pages: dict[int, str],
    keywords: dict[str, list[str]],
    *,
    slop: int = 1,
    edit_distance: int = 0,
    min_fuzzy_term_length: int = 99,
    keep_stopwords: bool = True,
    stem_words: bool = True,
    prefixlength: int = 0,
) -> pd.DataFrame:
    with tempfile.TemporaryDirectory() as temp_dir:
        root = Path(temp_dir)
        ocr_path = root / "ocr.json"
        keywords_path = root / "keywords.json"
        index_dir = root / "index"

        ocr_path.write_text(
            json.dumps(
                {
                    "encounters": [
                        {
                            "id": "demo_encounter",
                            "file_path": "demo.txt",
                            "text": make_encounter_text(pages),
                        }
                    ]
                }
            ),
            encoding="utf-8",
        )
        keywords_path.write_text(json.dumps(keywords), encoding="utf-8")

        settings = DetectionSettings(
            slop=slop,
            edit_distance=edit_distance,
            min_fuzzy_term_length=min_fuzzy_term_length,
            keep_stopwords=keep_stopwords,
            stem_words=stem_words,
            prefixlength=prefixlength,
            matched_only=False,
            include_file_path=False,
        )
        output = run_keyword_detection(
            ocr_json_file=ocr_path,
            keywords_json_file=keywords_path,
            settings=settings,
            index_dir=index_dir,
        )

    output_by_page = {record["page_number"]: record for record in output}
    rows = []
    for page_number, text in pages.items():
        record = output_by_page.get(page_number, {"matches": []})
        matches = record.get("matches", [])
        rows.append(
            {
                "page_number": page_number,
                "ocr_text": text,
                "detected_groups": ", ".join(match["group"] for match in matches),
                "detected_variants": ", ".join(match["variant"] for match in matches),
                "match_count": len(matches),
            }
        )
    return pd.DataFrame(rows)


def matched_pages(result_df: pd.DataFrame) -> list[int]:
    return result_df.loc[result_df["match_count"] > 0, "page_number"].tolist()


def compare_demo(gt_data: dict, method_data: list[dict], preferred_match_field: str = "group") -> pd.DataFrame:
    with tempfile.TemporaryDirectory() as temp_dir:
        root = Path(temp_dir)
        gt_dir = root / "gt"
        method_dir = root / "method"
        gt_dir.mkdir()
        method_dir.mkdir()
        declared_file_name = gt_data.get("file_name", "case.json") if isinstance(gt_data, dict) else "case.json"
        base_name = Path(str(declared_file_name)).stem or "case"
        (gt_dir / f"{base_name}.json").write_text(json.dumps(gt_data), encoding="utf-8")
        (method_dir / f"{base_name}_keyword_output.json").write_text(json.dumps(method_data), encoding="utf-8")

        metrics = compare_output_dir_vs_gt(
            method_output_dir=method_dir,
            gt_output_dir=gt_dir,
            method_glob="*_keyword_output.json",
            preferred_match_field=preferred_match_field,
        ).to_dict()

    important_columns = [
        "kw_tp", "kw_fp", "kw_fn", "kw_precision", "kw_recall", "kw_f1",
        "page_tp", "page_fp", "page_fn", "page_tn", "page_precision", "page_recall", "page_f1", "page_accuracy",
        "gt_pages_with_keywords_total", "method_pages_detected", "total_pages_all_docs",
    ]
    return pd.DataFrame([{column: metrics[column] for column in important_columns}])

## 3. Current Parameter Grid

The production tuning grid currently tests these values:

```text
slop: [1, 2, 3, 4, 5]
edit_distance: [1, 2]
min_fuzzy_term_length: [1, 2, 3, 4, 5, 6, 7, 8, 9]
keep_stopwords: [False, True]
stem_words: True
prefixlength: 0
```

That is `5 x 2 x 9 x 2 = 180` WHOOSH runs.

In [ ]:
current_grid = pd.DataFrame(
    [
        {"parameter": "slop", "values": [1, 2, 3, 4, 5]},
        {"parameter": "edit_distance", "values": [1, 2]},
        {"parameter": "min_fuzzy_term_length", "values": [1, 2, 3, 4, 5, 6, 7, 8, 9]},
        {"parameter": "keep_stopwords", "values": [False, True]},
        {"parameter": "stem_words", "values": [True]},
        {"parameter": "prefixlength", "values": [0]},
    ]
)
display(current_grid)
print("Total combinations:", 5 * 2 * 9 * 2)

## 4. Single-Word Keyword: `slop` Does Not Matter

`slop` controls distance between multiple words. If the keyword has only one word, there is no distance to measure.

In this example, `Emergency` is one word. `slop=1` and `slop=5` return the same page.

In [ ]:
single_word_pages = {
    1: "Contact Type: EMERGENCY",
    2: "Routine follow up visit",
}
single_word_keywords = {"emergent": ["Emergency"]}

rows = []
for slop in [1, 5]:
    result = run_demo(
        single_word_pages,
        single_word_keywords,
        slop=slop,
        edit_distance=0,
        min_fuzzy_term_length=99,
        keep_stopwords=True,
        stem_words=True,
    )
    rows.append({"slop": slop, "matched_pages": matched_pages(result)})

display(pd.DataFrame(rows))

## 5. Multi-Word Keyword: `slop`

`slop` means how close words must be to each other.

- `slop=1`: words must be right next to each other.
- `slop=2`: allows a small gap.
- `slop=5`: allows a wider gap.

The code uses unordered matching, so the words can be near each other even if OCR changes the order.

In [ ]:
slop_pages = {
    1: "signed by physician",
    2: "signed electronically by physician",
    3: "signed electronically after review by physician",
}
slop_keywords = {"signature": ["signed by"]}

summary_rows = []
for slop in [1, 2, 5]:
    result = run_demo(
        slop_pages,
        slop_keywords,
        slop=slop,
        edit_distance=0,
        min_fuzzy_term_length=99,
        keep_stopwords=True,
        stem_words=True,
    )
    summary_rows.append({"slop": slop, "matched_pages": matched_pages(result)})

display(pd.DataFrame(summary_rows))
display(result)

Expected interpretation:

- `slop=1` only catches page 1 because `signed by` is adjacent.
- `slop=2` catches pages 1 and 2 because there is one word between `signed` and `by` on page 2.
- `slop=5` catches pages 1, 2, and 3 because the phrase is allowed to be more spread out.

## 6. `edit_distance`: OCR Typo Tolerance

`edit_distance` controls how many character differences are allowed.

- `0`: exact word only.
- `1`: one character can be wrong, missing, or extra.
- `2`: up to two character differences.

This helps with OCR spelling mistakes.

In [ ]:
edit_distance_pages = {
    1: "Physician note signed",
    2: "Physican note signed",   # one missing letter
    3: "Phsician note signed",   # one missing letter near the beginning
    4: "Pysican note signed",    # farther from Physician
}
edit_distance_keywords = {"provider": ["Physician"]}

summary_rows = []
for edit_distance in [0, 1, 2]:
    result = run_demo(
        edit_distance_pages,
        edit_distance_keywords,
        slop=1,
        edit_distance=edit_distance,
        min_fuzzy_term_length=1,
        keep_stopwords=True,
        stem_words=True,
    )
    summary_rows.append({"edit_distance": edit_distance, "matched_pages": matched_pages(result)})

display(pd.DataFrame(summary_rows))
display(result)

Business interpretation:

- Higher `edit_distance` can recover more OCR typos.
- Higher `edit_distance` can also increase false positives if unrelated words are close enough.

## 7. `min_fuzzy_term_length`: Protect Short Words

`min_fuzzy_term_length` decides when typo matching is allowed.

If a keyword term is shorter than this value, it must match exactly.

This is important because short words are risky. Example: `STAT` is close to `START` and `STATE`.

In [ ]:
min_fuzzy_pages = {
    1: "STAT order",
    2: "START order",
    3: "STATE order",
}
min_fuzzy_keywords = {"urgent": ["STAT"]}

summary_rows = []
for min_fuzzy_term_length in [1, 5]:
    result = run_demo(
        min_fuzzy_pages,
        min_fuzzy_keywords,
        slop=1,
        edit_distance=1,
        min_fuzzy_term_length=min_fuzzy_term_length,
        keep_stopwords=True,
        stem_words=True,
    )
    summary_rows.append(
        {
            "min_fuzzy_term_length": min_fuzzy_term_length,
            "matched_pages": matched_pages(result),
        }
    )

display(pd.DataFrame(summary_rows))
display(result)

Expected interpretation:

- `min_fuzzy_term_length=1`: `STAT` can fuzzy match `START` and `STATE`.
- `min_fuzzy_term_length=5`: `STAT` has only 4 letters, so it must match exactly. This reduces false positives.

## 8. `keep_stopwords`: Keep or Ignore Common Words

Stop words are common words like `by`, `the`, `and`, `of`, `to`.

For a keyword like `signed by`, the word `by` is meaningful. Removing it can make the search too broad.

In [ ]:
stopword_pages = {
    1: "signed by physician",
    2: "signed yesterday by physician",
    3: "signed yesterday",
}
stopword_keywords = {"signature": ["signed by"]}

summary_rows = []
for keep_stopwords in [False, True]:
    result = run_demo(
        stopword_pages,
        stopword_keywords,
        slop=1,
        edit_distance=0,
        min_fuzzy_term_length=99,
        keep_stopwords=keep_stopwords,
        stem_words=True,
    )
    summary_rows.append({"keep_stopwords": keep_stopwords, "matched_pages": matched_pages(result)})

display(pd.DataFrame(summary_rows))
display(result)

Expected interpretation:

- `keep_stopwords=False`: `by` is removed, so the keyword behaves more like `signed`. This can match too many pages.
- `keep_stopwords=True`: `signed by` must appear as a phrase, so the result is stricter.

## 9. `stem_words`: Match Word Forms

Stemming normalizes related word forms.

Production tuning keeps this enabled.

Example: `review`, `reviewed`, and `reviewing` become similar searchable forms.

In [ ]:
stem_pages = {
    1: "review complete",
    2: "reviewed by nurse",
    3: "reviewing case",
}
stem_keywords = {"review_status": ["review"]}

summary_rows = []
for stem_words in [False, True]:
    result = run_demo(
        stem_pages,
        stem_keywords,
        slop=1,
        edit_distance=0,
        min_fuzzy_term_length=1,
        keep_stopwords=True,
        stem_words=stem_words,
    )
    summary_rows.append({"stem_words": stem_words, "matched_pages": matched_pages(result)})

display(pd.DataFrame(summary_rows))
display(result)

Expected interpretation:

- `stem_words=False`: only exact `review` matches.
- `stem_words=True`: `review`, `reviewed`, and `reviewing` can all match.

This is why the production workflow keeps stemming enabled.

## 10. `prefixlength`: Require the Beginning of a Word

The current tuning config uses `prefixlength=0`.

This means fuzzy matching does not require the first character to match exactly.

If `prefixlength=1`, the first character must match. That is stricter.

In [ ]:
prefix_pages = {
    1: "Physician note",
    2: "Xhysician note",  # first character is wrong
    3: "Physican note",   # missing letter later in the word
}
prefix_keywords = {"provider": ["Physician"]}

summary_rows = []
for prefixlength in [0, 1, 2]:
    result = run_demo(
        prefix_pages,
        prefix_keywords,
        slop=1,
        edit_distance=1,
        min_fuzzy_term_length=1,
        keep_stopwords=True,
        stem_words=True,
        prefixlength=prefixlength,
    )
    summary_rows.append({"prefixlength": prefixlength, "matched_pages": matched_pages(result)})

display(pd.DataFrame(summary_rows))
display(result)

Expected interpretation:

- `prefixlength=0`: can match beginning-character OCR errors like `Xhysician`.
- `prefixlength=1`: requires the first character to be correct, so `Xhysician` no longer matches.
- `Physican` still matches because the beginning is correct and the typo is later.

## 11. Page-Level vs Keyword-Level Metrics

Page-level metrics only ask: did this page have at least one keyword?

Keyword-level metrics ask: did this exact keyword label match on this exact page?

This example uses the same format as RapidFuzz and GT outputs.

In [ ]:
gt_example = {
    "file_name": "demo.json",
    "total_pages": 116,
    "pages_with_keywords": [
        {
            "page_number": 3,
            "keywords_detected": [
                {"keyword": "emergent", "reason": "Found Emergency variant"}
            ],
        },
        {
            "page_number": 5,
            "keywords_detected": [
                {"keyword": "immediate", "reason": "Found IMMEDIATELY variant"}
            ],
        },
        {"page_number": 9, "keywords_detected": []},
    ],
}

rapidfuzz_like_output = [
    {"page_number": 1, "encounter_id": 1, "matches": [], "file_path": ""},
    {"page_number": 2, "encounter_id": 2, "matches": [], "file_path": ""},
    {
        "page_number": 3,
        "encounter_id": 2,
        "matches": [{"group": "emergent", "variant": "Emergency"}],
        "file_path": "",
    },
    {"page_number": 4, "encounter_id": 2, "matches": [], "file_path": ""},
]

metrics_df = compare_demo(gt_example, rapidfuzz_like_output, preferred_match_field="group")
display(metrics_df.T.rename(columns={0: "value"}))

Expected interpretation:

- GT keyword pages are 3 and 5.
- Method detected page 3 only.
- Page-level: `TP=1`, `FP=0`, `FN=1`, `TN=114`.
- Keyword-level: `TP=1` for `emergent`, `FP=0`, `FN=1` for missed `immediate`.
- Page 9 has an empty keyword list, so it is not a positive page.

## 12. Optional: Run the Current 180-Combination Grid on Tiny OCR

This is a small demo grid, not the real business run.

It shows how the current 180 parameter combinations can lead to different detection behavior on a tiny OCR sample.

In [ ]:
tiny_pages = {
    1: "signed by physician",
    2: "signed electronically by physician",
    3: "Physican note signed",
    4: "START order",
}
tiny_keywords = {
    "signature": ["signed by"],
    "provider": ["Physician"],
    "urgent": ["STAT"],
}

grid_rows = []
for slop, edit_distance, min_fuzzy_term_length, keep_stopwords in product(
    [1, 2, 3, 4, 5],
    [1, 2],
    [1, 2, 3, 4, 5, 6, 7, 8, 9],
    [False, True],
):
    result = run_demo(
        tiny_pages,
        tiny_keywords,
        slop=slop,
        edit_distance=edit_distance,
        min_fuzzy_term_length=min_fuzzy_term_length,
        keep_stopwords=keep_stopwords,
        stem_words=True,
        prefixlength=0,
    )
    detected_groups = sorted(
        {
            group.strip()
            for value in result["detected_groups"]
            for group in value.split(",")
            if group.strip()
        }
    )
    grid_rows.append(
        {
            "slop": slop,
            "edit_distance": edit_distance,
            "min_fuzzy_term_length": min_fuzzy_term_length,
            "keep_stopwords": keep_stopwords,
            "matched_pages": matched_pages(result),
            "matched_page_count": len(matched_pages(result)),
            "detected_groups": ", ".join(detected_groups),
        }
    )

grid_df = pd.DataFrame(grid_rows)
print("Rows:", len(grid_df))
display(grid_df.head(20))
display(
    grid_df.groupby(["matched_page_count", "detected_groups"])
    .size()
    .reset_index(name="combination_count")
    .sort_values(["matched_page_count", "combination_count"], ascending=[False, False])
)

## 13. Business Takeaway

Parameter tuning is not changing the ground truth. It is changing how strict WHOOSH is when searching OCR text.

Strict settings usually reduce false positives but can miss OCR-damaged text.

Broad settings usually find more OCR-damaged text but can create false positives.

The final report chooses the best balance based on the configured ranking metric.